# LaBSE Indic Alignment Fine-tuning: 13-Model Directed-Pair Sweep

This notebook trains **13 separate LaBSE fine-tuned models** from `sentence-transformers/LaBSE`.

The 13 runs are:

1. **5 top-single models**: one model for each Top-5 directed pair, using 1000 examples per pair.
2. **5 bottom-single models**: one model for each Bottom-5 directed pair, using 1000 examples per pair.
3. **1 top5-all model**: one model trained on all Top-5 directed pairs.
4. **1 bottom5-all model**: one model trained on all Bottom-5 directed pairs.
5. **1 mixed model**: one model trained on Top-5 + Bottom-5 directed pairs.

This version intentionally **does not save rolling checkpoints**.  
It saves only:

- train/validation split CSVs,
- per-epoch training metrics CSVs,
- one `best_model/` folder per run,
- one `final_model/` folder per run,
- a verification CSV showing whether all 13 models were created.

## 1. Install dependencies

Run this cell first.  
In Colab, after installing, it is safest to do **Runtime → Restart runtime**, then continue from the imports cell.

The install cell pins `pandas`, `numpy`, and `scikit-learn` to avoid Colab dependency conflicts.

In [ ]:
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    _guard_dir = _candidate / "scripts"
    if (_guard_dir / "import_guard.py").exists():
        if str(_guard_dir) not in sys.path:
            sys.path.insert(0, str(_guard_dir))
        break
else:
    raise RuntimeError("Could not locate scripts/import_guard.py. Run this notebook from the WSAI workspace or copy the guard module alongside it.")

from import_guard import install_pandas_guards
install_pandas_guards()


In [ ]:
%pip -q install -U \
  "sentence-transformers" \
  "datasets" \
  "accelerate" \
  "transformers>=4.41,<5" \
  "huggingface_hub" \
  "tqdm" \
  "pandas==2.2.2" \
  "numpy==2.0.2" \
  "scikit-learn>=1.5,<1.9"

## 2. Imports, seed, device, and runtime setup

In [ ]:
import os

# Disable W&B prompts and reduce CUDA memory fragmentation.
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import math
import json
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from datasets import load_dataset
from sentence_transformers import SentenceTransformer, losses
from transformers import get_linear_schedule_with_warmup

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

IS_COLAB = "google.colab" in str(get_ipython())
IS_KAGGLE = os.path.exists("/kaggle/working")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = DEVICE

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2))

## 3. Project/output folder

For RunPod or other cloud GPU platforms with network volume, keep `/workspace`.

If you are using Colab and want Google Drive, set `USE_GOOGLE_DRIVE = True`.  
For a clean fresh run, set `CLEAR_OLD_PROJECT_CONTENTS = True` once.

In [ ]:
USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE and IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    PROJECT_DIR = Path("/content/drive/MyDrive/labse_13_model_directed_pair_finetuning")
elif os.path.exists("/workspace"):
    PROJECT_DIR = Path("/workspace/labse_13_model_directed_pair_finetuning")
elif IS_KAGGLE:
    PROJECT_DIR = Path("/kaggle/working/labse_13_model_directed_pair_finetuning")
else:
    PROJECT_DIR = Path("./labse_13_model_directed_pair_finetuning")

# WARNING: This deletes all previous results in PROJECT_DIR.
CLEAR_OLD_PROJECT_CONTENTS = False

if CLEAR_OLD_PROJECT_CONTENTS and PROJECT_DIR.exists():
    print("Deleting old project folder:", PROJECT_DIR)
    shutil.rmtree(PROJECT_DIR)

OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_DIR = PROJECT_DIR / "data"
METRICS_DIR = PROJECT_DIR / "metrics"

for p in [PROJECT_DIR, OUTPUT_DIR, DATA_DIR, METRICS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Output dir:", OUTPUT_DIR)
print("Metrics dir:", METRICS_DIR)

## 4. Hyperparameters

These settings are conservative for LaBSE so that we can check whether embeddings improve without destroying the original multilingual alignment.

For a 32GB GPU, `BATCH_SIZE = 32` is safe.  
If you get CUDA OOM, reduce it to `16`.

In [ ]:
BASE_MODEL_NAME = "sentence-transformers/LaBSE"

MAX_SEQ_LENGTH = 128
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-5
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0

# 1000 per directed pair, as requested.
TRAIN_EXAMPLES_PER_DIRECTED_PAIR = 1000

# Validation split for best model selection.
VAL_SIZE = 0.05
VAL_MAX_N = 1000

# Keep 0 for real run.
QUICK_TRAIN_N_PER_RUN = 0

# Evaluation batch size for validation metrics.
EVAL_BATCH_SIZE = 128

# AMP saves GPU memory.
USE_AMP = torch.cuda.is_available()

# Model-level resume: skips runs that already have both best_model and final_model.
# This is not checkpointing. It only prevents retraining a completed run.
SKIP_COMPLETED_RUNS = True

# Best model is selected using validation cosine gap.
BEST_MODEL_METRIC = "val_cosine_gap"

print("Base model:", BASE_MODEL_NAME)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Examples per directed pair:", TRAIN_EXAMPLES_PER_DIRECTED_PAIR)
print("Best model metric:", BEST_MODEL_METRIC)
print("USE_AMP:", USE_AMP)

## 5. Language mappings and selected directed pairs

In [ ]:
LANG_TO_IN22 = {
    "asm": "asm_Beng",
    "ben": "ben_Beng",
    "brx": "brx_Deva",
    "doi": "doi_Deva",
    "gom": "gom_Deva",
    "guj": "guj_Gujr",
    "hin": "hin_Deva",
    "kan": "kan_Knda",
    "kas": "kas_Arab",
    "mai": "mai_Deva",
    "mal": "mal_Mlym",
    "mar": "mar_Deva",
    "mni": "mni_Mtei",
    "npi": "npi_Deva",
    "ory": "ory_Orya",
    "pan": "pan_Guru",
    "san": "san_Deva",
    "sat": "sat_Olck",
    "snd": "snd_Deva",
    "tam": "tam_Taml",
    "tel": "tel_Telu",
    "urd": "urd_Arab",
    "eng": "eng_Latn",
}

TOP5_DIRECTED_PAIRS = [
    ("kan", "tel"),
    ("urd", "guj"),
    ("ben", "kan"),
    ("guj", "mar"),
    ("guj", "urd"),
]

BOTTOM5_DIRECTED_PAIRS = [
    ("sat", "npi"),
    ("sat", "urd"),
    ("sat", "hin"),
    ("kan", "sat"),
    ("ben", "mni"),
]

def direction_name(src, tgt):
    return f"{src}→{tgt}"

def direction_label(src, tgt):
    return f"{src} → {tgt}"

def pair_slug(src, tgt):
    return f"{src}_to_{tgt}"

def resolve_sentence_col(short_lang, df):
    if short_lang not in LANG_TO_IN22:
        raise KeyError(f"Unknown language code: {short_lang}")

    code = LANG_TO_IN22[short_lang]

    candidates = [
        code,
        f"sentence_{code}",
        short_lang,
        f"sentence_{short_lang}",
    ]

    if short_lang == "snd":
        candidates += [
            "snd_Deva",
            "sentence_snd_Deva",
            "snd_Arab",
            "sentence_snd_Arab",
        ]

    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError(
        f"No matching column found for language={short_lang}, code={code}.\n"
        f"Tried: {candidates}\n"
        f"Available columns: {list(df.columns)}"
    )

## 6. Create the 13 run specifications

This cell verifies that exactly 13 model runs are defined:

- 5 top-single
- 5 bottom-single
- 1 top5-all
- 1 bottom5-all
- 1 mixed top+bottom

In [ ]:
RUN_SPECS = []

for i, (src, tgt) in enumerate(TOP5_DIRECTED_PAIRS, start=1):
    RUN_SPECS.append({
        "run_name": f"top_single_{i}_{pair_slug(src, tgt)}",
        "run_type": "top_single",
        "directed_pairs": [(src, tgt)],
    })

for i, (src, tgt) in enumerate(BOTTOM5_DIRECTED_PAIRS, start=1):
    RUN_SPECS.append({
        "run_name": f"bottom_single_{i}_{pair_slug(src, tgt)}",
        "run_type": "bottom_single",
        "directed_pairs": [(src, tgt)],
    })

RUN_SPECS.append({
    "run_name": "top5_all_directed_pairs",
    "run_type": "top5_all",
    "directed_pairs": TOP5_DIRECTED_PAIRS,
})

RUN_SPECS.append({
    "run_name": "bottom5_all_directed_pairs",
    "run_type": "bottom5_all",
    "directed_pairs": BOTTOM5_DIRECTED_PAIRS,
})

RUN_SPECS.append({
    "run_name": "mixed_top5_bottom5_directed_pairs",
    "run_type": "mixed_top_bottom",
    "directed_pairs": TOP5_DIRECTED_PAIRS + BOTTOM5_DIRECTED_PAIRS,
})

run_spec_df = pd.DataFrame([
    {
        "run_name": spec["run_name"],
        "run_type": spec["run_type"],
        "num_directions": len(spec["directed_pairs"]),
        "directions": ", ".join(direction_label(*p) for p in spec["directed_pairs"]),
        "expected_examples_before_validation": len(spec["directed_pairs"]) * TRAIN_EXAMPLES_PER_DIRECTED_PAIR,
    }
    for spec in RUN_SPECS
])

assert len(RUN_SPECS) == 13, f"Expected 13 runs, got {len(RUN_SPECS)}"
assert run_spec_df["run_name"].nunique() == 13, "Run names are not unique"

print("Verified run count:", len(RUN_SPECS))
display(run_spec_df)

run_spec_df.to_csv(METRICS_DIR / "run_specifications_13_models.csv", index=False)

## 7. Load IN22-Gen training source

This notebook uses **IN22-Gen** as the training source for the 13-model sweep because the requested top/bottom pairs come from IN22-style aligned columns.

For final research reporting, keep a separate held-out evaluation such as **IN22-Conv** for before/after evaluation.

In [ ]:
def load_in22_dataset_as_dataframe(dataset_name, dataset_config="default", preferred_splits=("gen", "train", "test", "validation")):
    try:
        if dataset_config is None or dataset_config == "":
            dataset_obj = load_dataset(dataset_name)
        else:
            dataset_obj = load_dataset(dataset_name, dataset_config)
    except Exception as e:
        raise RuntimeError(
            f"Could not load {dataset_name} with config={dataset_config!r}. "
            f"Original error: {e}"
        ) from e

    if hasattr(dataset_obj, "keys"):
        available_splits = list(dataset_obj.keys())
        chosen_split = None

        for split in preferred_splits:
            if split in available_splits:
                chosen_split = split
                break

        if chosen_split is None:
            chosen_split = available_splits[0]

        print(f"Available splits for {dataset_name}: {available_splits}")
        print(f"Using split for {dataset_name}: {chosen_split}")
        return dataset_obj[chosen_split].to_pandas()

    return dataset_obj.to_pandas()

in22_gen_df_raw = load_in22_dataset_as_dataframe(
    "ai4bharat/IN22-Gen",
    "default",
    preferred_splits=("gen", "train", "test", "validation"),
)

print("Raw IN22-Gen shape:", in22_gen_df_raw.shape)
print("Columns:", in22_gen_df_raw.columns.tolist())

needed_langs = sorted(set([
    lang
    for spec in RUN_SPECS
    for pair in spec["directed_pairs"]
    for lang in pair
]))

SENTENCE_COLS = {
    lang: resolve_sentence_col(lang, in22_gen_df_raw)
    for lang in needed_langs
}

print("\nResolved sentence columns:")
for lang, col in SENTENCE_COLS.items():
    print(f"{lang:4s} -> {col}")

## 8. Data-building utilities

In [ ]:
def sentence_col(short_lang):
    if short_lang not in SENTENCE_COLS:
        raise KeyError(f"{short_lang} not found in SENTENCE_COLS. Available: {list(SENTENCE_COLS.keys())}")
    return SENTENCE_COLS[short_lang]

def make_run_dirs(run_name):
    run_base_dir = OUTPUT_DIR / run_name
    data_dir = run_base_dir / "data"
    best_model_dir = run_base_dir / "best_model"
    final_model_dir = run_base_dir / "final_model"
    metrics_dir = run_base_dir / "metrics"

    for p in [run_base_dir, data_dir, best_model_dir, final_model_dir, metrics_dir]:
        p.mkdir(parents=True, exist_ok=True)

    return {
        "run_base_dir": run_base_dir,
        "data_dir": data_dir,
        "best_model_dir": best_model_dir,
        "final_model_dir": final_model_dir,
        "metrics_dir": metrics_dir,
    }

def build_training_df_for_directed_pairs(in22_df, directed_pairs, run_name):
    records = []

    for pair_index, (src, tgt) in enumerate(directed_pairs):
        src_col = sentence_col(src)
        tgt_col = sentence_col(tgt)

        keep_cols = [src_col, tgt_col]
        if "id" in in22_df.columns:
            keep_cols = ["id"] + keep_cols

        pair_df = in22_df[keep_cols].copy()
        pair_df = pair_df.rename(columns={src_col: "sentence1", tgt_col: "sentence2"})

        pair_df["run_name"] = run_name
        pair_df["source_language"] = src
        pair_df["target_language"] = tgt
        pair_df["direction"] = direction_name(src, tgt)
        pair_df["pair"] = direction_label(src, tgt)

        pair_df["sentence1"] = pair_df["sentence1"].astype(str).str.strip()
        pair_df["sentence2"] = pair_df["sentence2"].astype(str).str.strip()

        pair_df = pair_df[
            pair_df["sentence1"].ne("") &
            pair_df["sentence2"].ne("") &
            pair_df["sentence1"].str.lower().ne("nan") &
            pair_df["sentence2"].str.lower().ne("nan")
        ].copy()

        pair_df = pair_df.drop_duplicates(subset=["sentence1", "sentence2"]).reset_index(drop=True)

        if TRAIN_EXAMPLES_PER_DIRECTED_PAIR and TRAIN_EXAMPLES_PER_DIRECTED_PAIR > 0:
            pair_df = pair_df.sample(
                n=min(TRAIN_EXAMPLES_PER_DIRECTED_PAIR, len(pair_df)),
                random_state=SEED + pair_index,
            ).reset_index(drop=True)

        records.append(pair_df)

    if not records:
        raise ValueError(f"No records were created for run: {run_name}")

    out = pd.concat(records, ignore_index=True)

    if QUICK_TRAIN_N_PER_RUN and QUICK_TRAIN_N_PER_RUN > 0:
        out = out.sample(
            n=min(QUICK_TRAIN_N_PER_RUN, len(out)),
            random_state=SEED,
        ).reset_index(drop=True)

    return out

def split_train_val(run_df):
    if len(run_df) < 100:
        raise ValueError("Training dataframe is too small.")

    stratify_col = None
    direction_counts = run_df["direction"].value_counts()

    # Stratify only if there are multiple directions and each has enough rows.
    if run_df["direction"].nunique() > 1 and direction_counts.min() >= 2:
        stratify_col = run_df["direction"]

    train_df, val_df = train_test_split(
        run_df,
        test_size=VAL_SIZE,
        random_state=SEED,
        shuffle=True,
        stratify=stratify_col,
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    if VAL_MAX_N and len(val_df) > VAL_MAX_N:
        val_df = val_df.sample(n=VAL_MAX_N, random_state=SEED).reset_index(drop=True)

    return train_df, val_df

## 9. Training dataset and validation metrics utilities

In [ ]:
class PairTextDataset(Dataset):
    def __init__(self, df):
        self.sentence1 = df["sentence1"].tolist()
        self.sentence2 = df["sentence2"].tolist()

    def __len__(self):
        return len(self.sentence1)

    def __getitem__(self, idx):
        return self.sentence1[idx], self.sentence2[idx]

def make_pair_dataloader(model, train_df, batch_size):
    dataset = PairTextDataset(train_df)

    def collate_fn(batch):
        texts1 = [x[0] for x in batch]
        texts2 = [x[1] for x in batch]
        sentence_features = [
            model.tokenize(texts1),
            model.tokenize(texts2),
        ]
        labels = torch.zeros(len(batch), dtype=torch.long)
        return sentence_features, labels

    return DataLoader(
        dataset,
        shuffle=True,
        batch_size=batch_size,
        drop_last=True,
        collate_fn=collate_fn,
    )

def move_features_to_device(sentence_features, labels, device):
    moved_features = []
    for feature in sentence_features:
        moved_features.append({
            key: value.to(device) if hasattr(value, "to") else value
            for key, value in feature.items()
        })
    labels = labels.to(device)
    return moved_features, labels

def stable_offset(n, seed, key):
    if n <= 1:
        return 0
    rng = np.random.default_rng(abs(hash((seed, key))) % (2**32))
    return int(rng.integers(1, n))

def evaluate_on_pairs(model, val_df, model_name, run_name):
    if len(val_df) == 0:
        return {}

    s1 = val_df["sentence1"].tolist()
    s2 = val_df["sentence2"].tolist()

    emb1 = model.encode(
        s1,
        batch_size=EVAL_BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )

    emb2 = model.encode(
        s2,
        batch_size=EVAL_BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )

    sim = emb1 @ emb2.T
    n = sim.shape[0]

    src2trg_acc = float(np.mean(np.argmax(sim, axis=1) == np.arange(n)))
    trg2src_acc = float(np.mean(np.argmax(sim, axis=0) == np.arange(n)))
    mean_acc = (src2trg_acc + trg2src_acc) / 2.0

    gold_cos = np.diag(sim)

    offset = stable_offset(n, SEED, f"{model_name}_{run_name}")
    random_cos = np.sum(emb1 * np.roll(emb2, shift=offset, axis=0), axis=1)

    mean_gold = float(np.mean(gold_cos))
    sd_gold = float(np.std(gold_cos))
    mean_random = float(np.mean(random_cos))
    sd_random = float(np.std(random_cos))
    cosine_gap = mean_gold - mean_random
    gap_sd = float(np.std(gold_cos - random_cos))

    midpoint = (mean_gold + mean_random) / 2.0
    sensitivity_midpoint = float(np.mean(gold_cos >= midpoint))
    specificity_midpoint = float(np.mean(random_cos < midpoint))
    balanced_accuracy_midpoint = (sensitivity_midpoint + specificity_midpoint) / 2.0

    return {
        "val_retrieval_src2trg_accuracy": src2trg_acc,
        "val_retrieval_trg2src_accuracy": trg2src_acc,
        "val_retrieval_mean_accuracy": mean_acc,
        "val_mean_gold_cosine": mean_gold,
        "val_sd_gold_cosine": sd_gold,
        "val_mean_random_cosine": mean_random,
        "val_sd_random_cosine": sd_random,
        "val_cosine_gap": cosine_gap,
        "val_sd_cosine_gap": gap_sd,
        "val_sensitivity_midpoint": sensitivity_midpoint,
        "val_specificity_midpoint": specificity_midpoint,
        "val_balanced_accuracy_midpoint": balanced_accuracy_midpoint,
        "val_midpoint_threshold": float(midpoint),
    }

## 10. Train one run

This function trains one LaBSE model from the base checkpoint.

It saves:

- `data/train_pairs_used.csv`
- `data/val_pairs_used.csv`
- `metrics/train_metrics.csv`
- `best_model/`
- `final_model/`

It does **not** save rolling checkpoints.

In [ ]:
def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def train_one_run(spec):
    run_name = spec["run_name"]
    directed_pairs = spec["directed_pairs"]
    run_dirs = make_run_dirs(run_name)

    best_model_dir = run_dirs["best_model_dir"]
    final_model_dir = run_dirs["final_model_dir"]
    metrics_path = run_dirs["metrics_dir"] / "train_metrics.csv"

    if SKIP_COMPLETED_RUNS:
        best_ready = (best_model_dir / "modules.json").exists()
        final_ready = (final_model_dir / "modules.json").exists()
        if best_ready and final_ready and metrics_path.exists():
            print(f"[SKIP] Completed run already exists: {run_name}")
            return pd.read_csv(metrics_path)

    print("\n" + "#" * 100)
    print("STARTING RUN:", run_name)
    print("Run type:", spec["run_type"])
    print("Directed pairs:", [direction_label(*p) for p in directed_pairs])
    print("#" * 100)

    # Build data
    run_df = build_training_df_for_directed_pairs(
        in22_df=in22_gen_df_raw,
        directed_pairs=directed_pairs,
        run_name=run_name,
    )

    train_df, val_df = split_train_val(run_df)

    train_df.to_csv(run_dirs["data_dir"] / "train_pairs_used.csv", index=False)
    val_df.to_csv(run_dirs["data_dir"] / "val_pairs_used.csv", index=False)
    run_df.to_csv(run_dirs["data_dir"] / "all_pairs_before_split.csv", index=False)

    print("Total examples:", len(run_df))
    print("Train examples:", len(train_df))
    print("Validation examples:", len(val_df))
    display(run_df.groupby(["source_language", "target_language", "pair"]).size().reset_index(name="n"))

    # Load fresh base model for each run
    model = SentenceTransformer(BASE_MODEL_NAME, device=DEVICE)
    model.max_seq_length = MAX_SEQ_LENGTH

    train_dataloader = make_pair_dataloader(model, train_df, BATCH_SIZE)
    steps_per_epoch = len(train_dataloader)

    if steps_per_epoch == 0:
        raise ValueError(f"No training batches for {run_name}. Reduce BATCH_SIZE or add more data.")

    total_steps = steps_per_epoch * EPOCHS
    warmup_steps = math.ceil(total_steps * WARMUP_RATIO)

    train_loss = losses.MultipleNegativesRankingLoss(model)
    train_loss.to(DEVICE)

    optimizer = torch.optim.AdamW(train_loss.parameters(), lr=LEARNING_RATE)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    config = {
        "run_name": run_name,
        "run_type": spec["run_type"],
        "base_model_name": BASE_MODEL_NAME,
        "directed_pairs": [direction_label(*p) for p in directed_pairs],
        "max_seq_length": MAX_SEQ_LENGTH,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "warmup_ratio": WARMUP_RATIO,
        "warmup_steps": warmup_steps,
        "steps_per_epoch": steps_per_epoch,
        "total_steps": total_steps,
        "train_examples": len(train_df),
        "validation_examples": len(val_df),
        "use_amp": USE_AMP,
        "best_model_metric": BEST_MODEL_METRIC,
        "best_model_dir": str(best_model_dir),
        "final_model_dir": str(final_model_dir),
        "note": "No rolling checkpoints are saved. Only best_model and final_model are stored.",
    }
    save_json(config, run_dirs["run_base_dir"] / "training_config.json")

    metrics_rows = []
    best_score = -float("inf")
    best_epoch = None

    print("Steps per epoch:", steps_per_epoch)
    print("Total steps:", total_steps)
    print("Warmup steps:", warmup_steps)
    print("Best model folder:", best_model_dir)
    print("Final model folder:", final_model_dir)

    global_step = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss.train()

        epoch_losses = []
        progress = tqdm(train_dataloader, desc=f"{run_name} | epoch {epoch}/{EPOCHS}")

        for sentence_features, labels in progress:
            global_step += 1

            sentence_features, labels = move_features_to_device(sentence_features, labels, DEVICE)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                loss_value = train_loss(sentence_features, labels)

            scaler.scale(loss_value).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(train_loss.parameters(), MAX_GRAD_NORM)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            loss_float = float(loss_value.detach().cpu().item())
            epoch_losses.append(loss_float)

            progress.set_postfix({
                "loss": f"{loss_float:.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}",
            })

        mean_train_loss = float(np.mean(epoch_losses)) if epoch_losses else np.nan

        model.eval()
        with torch.no_grad():
            val_metrics = evaluate_on_pairs(
                model=model,
                val_df=val_df,
                model_name=run_name,
                run_name=run_name,
            )

        row = {
            "run_name": run_name,
            "run_type": spec["run_type"],
            "epoch": epoch,
            "global_step": global_step,
            "mean_train_loss": mean_train_loss,
            "learning_rate": float(scheduler.get_last_lr()[0]),
            "best_model_metric": BEST_MODEL_METRIC,
            **val_metrics,
        }

        current_score = row.get(BEST_MODEL_METRIC, -float("inf"))
        is_best = current_score > best_score
        row["is_best_epoch"] = bool(is_best)

        if is_best:
            best_score = current_score
            best_epoch = epoch

            # Overwrite previous best model. This is not a checkpoint history.
            if best_model_dir.exists():
                shutil.rmtree(best_model_dir)
            model.save(str(best_model_dir))
            print(f"[BEST] {run_name}: epoch={epoch}, {BEST_MODEL_METRIC}={best_score:.6f}")

        metrics_rows.append(row)
        metrics_df = pd.DataFrame(metrics_rows)
        metrics_df.to_csv(metrics_path, index=False)

        print(
            f"Epoch {epoch}/{EPOCHS} | "
            f"loss={mean_train_loss:.4f} | "
            f"val_gap={row.get('val_cosine_gap', np.nan):.4f} | "
            f"val_retrieval_mean_acc={row.get('val_retrieval_mean_accuracy', np.nan):.4f}"
        )

    # Save final model after last epoch.
    if final_model_dir.exists():
        shutil.rmtree(final_model_dir)
    model.save(str(final_model_dir))

    final_info = {
        "run_name": run_name,
        "best_epoch": best_epoch,
        "best_score": best_score,
        "best_model_metric": BEST_MODEL_METRIC,
        "best_model_dir": str(best_model_dir),
        "final_model_dir": str(final_model_dir),
    }
    save_json(final_info, run_dirs["run_base_dir"] / "final_run_info.json")

    print("[DONE]", run_name)
    print("Best epoch:", best_epoch)
    print("Best score:", best_score)
    print("Saved best model:", best_model_dir)
    print("Saved final model:", final_model_dir)

    # Cleanup GPU memory before next run.
    del model, train_loss, optimizer, scheduler, scaler, train_dataloader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(metrics_rows)

## 11. Train all 13 models

This cell trains all 13 models sequentially.

If a run already has both `best_model/modules.json` and `final_model/modules.json`, it is skipped when `SKIP_COMPLETED_RUNS = True`.

In [ ]:
all_metrics = []

for spec in RUN_SPECS:
    metrics_df = train_one_run(spec)
    all_metrics.append(metrics_df)

all_metrics_df = pd.concat(all_metrics, ignore_index=True)
all_metrics_path = METRICS_DIR / "all_13_runs_train_metrics.csv"
all_metrics_df.to_csv(all_metrics_path, index=False)

print("Saved all training metrics to:", all_metrics_path)
display(all_metrics_df.tail())

## 12. Verify that all 13 models were generated

This cell checks that every run has:

- `best_model/modules.json`
- `final_model/modules.json`
- `metrics/train_metrics.csv`
- `training_config.json`

In [ ]:
verification_rows = []

for spec in RUN_SPECS:
    run_name = spec["run_name"]
    run_dirs = make_run_dirs(run_name)

    row = {
        "run_name": run_name,
        "run_type": spec["run_type"],
        "num_directions": len(spec["directed_pairs"]),
        "directions": ", ".join(direction_label(*p) for p in spec["directed_pairs"]),
        "best_model_exists": (run_dirs["best_model_dir"] / "modules.json").exists(),
        "final_model_exists": (run_dirs["final_model_dir"] / "modules.json").exists(),
        "metrics_exists": (run_dirs["metrics_dir"] / "train_metrics.csv").exists(),
        "config_exists": (run_dirs["run_base_dir"] / "training_config.json").exists(),
        "best_model_dir": str(run_dirs["best_model_dir"]),
        "final_model_dir": str(run_dirs["final_model_dir"]),
    }
    verification_rows.append(row)

verification_df = pd.DataFrame(verification_rows)
verification_df["complete"] = (
    verification_df["best_model_exists"] &
    verification_df["final_model_exists"] &
    verification_df["metrics_exists"] &
    verification_df["config_exists"]
)

verification_path = METRICS_DIR / "model_generation_verification_13_runs.csv"
verification_df.to_csv(verification_path, index=False)

print("Total run specs:", len(RUN_SPECS))
print("Completed models:", int(verification_df["complete"].sum()), "/", len(verification_df))
print("Verification saved to:", verification_path)

display(verification_df)

assert len(verification_df) == 13, f"Expected 13 rows, got {len(verification_df)}"
assert verification_df["complete"].all(), "Not all 13 runs completed successfully. Check verification_df."
print("✅ Verified: all 13 models generated successfully.")

## 13. Optional: zip only important results

This skips intermediate caches and stores:

- all `best_model/` folders,
- all `final_model/` folders,
- all CSV/JSON metrics/config files.

Use this before deleting a pod or network volume.

In [ ]:
import zipfile

EXPORT_DIR = PROJECT_DIR / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

zip_path = EXPORT_DIR / "labse_13_model_finetuning_results.zip"
keep_suffixes = {".csv", ".json", ".txt", ".png", ".jpg", ".jpeg", ".xlsx"}

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in PROJECT_DIR.rglob("*"):
        if not file.is_file():
            continue

        path_str = str(file)

        # Avoid nesting zip files.
        if "/exports/" in path_str and file.suffix == ".zip":
            continue

        should_keep = (
            "/best_model/" in path_str
            or "/final_model/" in path_str
            or file.suffix.lower() in keep_suffixes
        )

        if should_keep:
            z.write(file, arcname=file.relative_to(PROJECT_DIR))

print("Saved zip:", zip_path)
print("Zip size GB:", round(zip_path.stat().st_size / (1024 ** 3), 2))

## 14. How to use the generated models later

Each run produces two model folders:

```text
outputs/<run_name>/best_model
outputs/<run_name>/final_model
```

For evaluation, use `best_model` first because it is selected using validation cosine gap.

Example:

```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("/workspace/labse_13_model_directed_pair_finetuning/outputs/bottom5_all_directed_pairs/best_model")
```